In [0]:
from pyspark.sql.functions import (
    col, round, concat_ws, to_date,
    when, trim, upper, current_timestamp
)

od  = spark.read.table("bi.bronze.order_details").alias("od")
o   = spark.read.table("bi.bronze.orders").alias("o")
c   = spark.read.table("bi.bronze.customers").alias("c")
p   = spark.read.table("bi.bronze.products").alias("p")
cat = spark.read.table("bi.bronze.categories").alias("cat")
emp = spark.read.table("bi.bronze.employees").alias("emp")

silver_df = (
    od
    .join(o, col("od.order_id") == col("o.order_id"), "left")
    .join(c, col("o.customer_id") == col("c.customer_id"), "left")
    .join(p, col("od.product_id") == col("p.product_id"), "left")
    .join(cat, col("p.category_id") == col("cat.category_id"), "left")
    .join(emp, col("o.employee_id") == col("emp.employee_id"), "left")


    .withColumn("order_date_clean", to_date(col("o.order_date")))

    .withColumn("customer_country", trim(upper(col("c.country"))))
    .withColumn("ship_country_clean", trim(upper(col("o.ship_country"))))
    .withColumn("category_name_clean", trim(col("cat.category_name")))
    .withColumn("employee_name", concat_ws(" ", trim(col("emp.first_name")), trim(col("emp.last_name"))))

    .withColumn(
        "discount_clean",
        when((col("od.discount") < 0) | (col("od.discount") > 1), 0.0)
        .otherwise(col("od.discount"))
    )

    .filter(col("od.quantity") > 0)
    .filter(col("od.unit_price") > 0)

    .withColumn(
        "net_revenue",
        round(col("od.unit_price") * col("od.quantity") * (1 - col("discount_clean")), 2)
    )

        .select(
        col("o.order_id").alias("order_id"),
        col("order_date_clean").alias("order_date"),
        col("o.customer_id").alias("customer_id"),
        col("c.company_name").alias("customer_name"),
        col("customer_country"),
        col("c.city").alias("customer_city"),
        col("employee_name"),
        col("p.product_id").alias("product_id"),
        col("p.product_name").alias("product_name"),
        col("category_name_clean").alias("category_name"),
        col("od.unit_price").alias("unit_price"),
        col("od.quantity").alias("quantity"),
        col("discount_clean").alias("discount"),
        col("net_revenue"),
        col("o.freight").alias("freight"),
        col("ship_country_clean").alias("ship_country"),
        current_timestamp().alias("_processed_at")
    )
    .dropDuplicates(["order_id", "product_id"])
    .dropna(subset=["order_id", "customer_id", "product_id"])
)

total = od.count()
after_clean = silver_df.count()
dropped = total - after_clean

print(f"[SILVER] Rekordy wejściowe: {total}")
print(f"[SILVER] Rekordy po czyszczeniu: {after_clean}")
print(f"[SILVER] Odrzucone rekordy: {dropped} ({dropped/total*100:.1f}%)")

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bi.silver.orders_denormalized")
)

print(f"[SILVER] orders_denormalized: {silver_df.count()} rekordów")